# Ajustement du modèle pidgin (Colab)

1. Mets le fichier **`KAMA_pidgin_drive.zip` tel quel** dans **Mon Drive**, sans le décompresser : la cellule 2 s'en charge.
2. Ouvre ce notebook dans Colab (Fichier → Importer le notebook, ou double-clic dans le Drive une fois décompressé).
3. **Exécution → Modifier le type d'exécution → GPU T4**.
4. Lance les cellules dans l'ordre. Si Colab coupe, relance tout : l'entraînement reprend au dernier point sauvegardé.

## 1. Réglages
`MON_DRIVE` correspond à **Mon Drive** dans l'interface de Google Drive. Ne change `DOSSIER` que si tu ranges le travail ailleurs.

In [ ]:
MON_DRIVE = '/content/drive/MyDrive'
ZIP = MON_DRIVE + '/KAMA_pidgin_drive.zip'
DOSSIER = MON_DRIVE + '/KAMA/pidgin'
EPOCHS = 8

## 2. Drive et données

In [ ]:
import os, zipfile
from google.colab import drive
drive.mount('/content/drive')
if not os.path.exists(DOSSIER + '/data/train.jsonl'):
    if not os.path.exists(ZIP):
        raise FileNotFoundError(f'Mets KAMA_pidgin_drive.zip dans Mon Drive : {ZIP} introuvable')
    with zipfile.ZipFile(ZIP) as z:
        z.extractall(MON_DRIVE)  # cree Mon Drive/KAMA/pidgin
for f in ('data/train.jsonl', 'data/dev.jsonl', '02_entrainer.py'):
    assert os.path.exists(f'{DOSSIER}/{f}'), f'manquant : {f}'
print('Données prêtes :', sum(1 for _ in open(DOSSIER + '/data/train.jsonl')), 'exemples')

## 3. Installation

In [ ]:
!pip install -q transformers==4.57.6 sentencepiece sacremoses sacrebleu accelerate ctranslate2
!nvidia-smi -L

## 4. Entraînement
Environ 30 à 60 minutes sur un GPU T4. Le modèle et ses points de reprise sont écrits dans `Mon Drive/KAMA/pidgin/modele_ajuste`.

In [ ]:
%cd {DOSSIER}
!python 02_entrainer.py --data data --sortie modele_ajuste --epochs {EPOCHS}

## 5. Conversion pour le serveur
Produit `modele_ajuste/pidgin_en_wes.zip` (environ 80 Mo). Télécharge-le, décompresse-le dans `KAMA/models/poids/ct2/` sur ton poste : il remplace le modèle du serveur, qui garde le même nom.

In [ ]:
!ct2-transformers-converter --model modele_ajuste/final --output_dir modele_ajuste/pidgin_en_wes --quantization int8 --copy_files source.spm target.spm --force
!cd modele_ajuste && zip -qr pidgin_en_wes.zip pidgin_en_wes && ls -la pidgin_en_wes.zip